# VideoTool — FULL RENDER on Kaggle (no whisper)

Renders a whole audio-story episode on the T4 GPU (NVENC). Claude Code CLI authors `creative.yaml`
(music + SFX + mood/overlay + description) and stages it on Drive; this notebook just renders.
**No LLM runs here.** Validated 2026-07-11 (ĐẠO SĨ 51-min ep, ~41 min wall, 1.085 GiB, QA-pass).

**One-time setup:**
1. On your machine: `base64 -w0 ~/.config/rclone/rclone.conf` -> copy the single line.
2. Kaggle: Add-ons -> Secrets -> add `RCLONE_CONF` = that base64 line, toggle **Attached**.
3. Right panel -> Session options -> Accelerator = **GPU T4 x2** (NOT None / P100 — P100 has no NVENC).
4. Save Version -> **Save & Run All (Commit)**. Rerun after a disconnect -> resumes from the Drive checkpoint.

Claude Code CLI fills in the paths in the last cell + pushes this kernel per episode.

In [ ]:
import os, subprocess, base64
from kaggle_secrets import UserSecretsClient
raw = UserSecretsClient().get_secret('RCLONE_CONF').strip()
try:
    conf = base64.b64decode(raw, validate=True).decode('utf-8'); assert '[gdrive]' in conf
except Exception:
    conf = raw
os.makedirs(os.path.expanduser('~/.config/rclone'), exist_ok=True)
open(os.path.expanduser('~/.config/rclone/rclone.conf'), 'w').write(conf)
subprocess.run('command -v rclone >/dev/null || (curl -s https://rclone.org/install.sh | sudo bash)', shell=True)
remotes = subprocess.run(['rclone', 'listremotes'], capture_output=True, text=True).stdout.strip()
print('rclone remotes:', remotes or '(NONE)', '| conf has [gdrive]:', '[gdrive]' in conf)
assert 'gdrive:' in remotes.split(), 'RCLONE_CONF has no [gdrive] remote (set it to base64 of rclone.conf).'
SHARED = 'gdrive:_VIDEOTOOL_SHARED'
for mod in ('videotool_cloud.py', 'cloud_director.py', 'cloud_render_runner.py'):
    subprocess.run(['rclone', 'copyto', f'{SHARED}/{mod}', mod], check=True)
for lib in ('sfx', 'overlays'):
    dst = os.path.expanduser(f'~/.local/share/videotool/{lib}')
    os.makedirs(dst, exist_ok=True)
    subprocess.run(['rclone', 'copy', f'{SHARED}/{lib}', dst, '--fast-list'], check=False)
import cloud_render_runner as rr
print('setup OK -> ready to render')

In [ ]:
# Claude Code CLI fills these per episode. For a folder under a non-default Google account (u/1 etc.)
# use the connection string form: gdrive,root_folder_id=<FOLDER_ID>:
SOURCE     = 'gdrive,root_folder_id=<FOLDER_ID>:'
OUTPUT     = 'gdrive,root_folder_id=<FOLDER_ID>:Output'
CHECKPOINT = 'gdrive:_VIDEOTOOL_SHARED/checkpoints/<EPISODE-SLUG>'
CREATIVE   = 'gdrive:_VIDEOTOOL_SHARED/creative/<EPISODE-SLUG>.yaml'
REPO_REF   = 'git+https://github.com/pnd4189/video-tool@<PUSHED_SHA>'   # MUST be a pushed commit

rr.render_job(SOURCE, OUTPUT, CHECKPOINT, creative_remote=CREATIVE, repo_ref=REPO_REF, local_job='/tmp/job')